# 01 - Business Understanding & Data Overview

**Objective:** frame the business problem, identify the available data and its grain, and define the analytical questions and KPIs.

## Business context
Capital Bikeshare (Washington D.C.) rents bicycles from a network of docking stations. The operator needs to know **when and under what conditions bikes are rented**, so it can plan bicycle availability, maintenance, staffing and capacity.

**Primary question:** *What factors and usage patterns are associated with rental demand, and how can the operator use them to improve service availability and operational planning?*

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore", message="findfont")
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from src.visualization.style import apply_matplotlib_style
apply_matplotlib_style()
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.max_columns", 40)

## Available files
The original archive `data/bike+sharing+dataset.zip` is extracted (unchanged) into `data/raw/`.

In [2]:
from src.data.ingest import extract_raw, load_raw
from src.config import RAW_DIR
print(extract_raw())
print((RAW_DIR / "Readme.txt").read_text()[2400:5200])

['day.csv', 'hour.csv', 'Readme.txt']
ronmental and seasonal settings.
	
	- Event and Anomaly Detection:  
		Count of rented bikes are also correlated to some events in the town which easily are traceable via search engines.
		For instance, query like "2012-10-30 washington d.c." in Google returns related results to Hurricane Sandy. Some of the important events are 
		identified in [1]. Therefore the data can be used for validation of anomaly or event detection algorithms as well.


Files

	- Readme.txt
	- hour.csv : bike sharing counts aggregated on hourly basis. Records: 17379 hours
	- day.csv - bike sharing counts aggregated on daily basis. Records: 731 days

	
Dataset characteristics
Both hour.csv and day.csv have the following fields, except hr which is not available in day.csv
	
	- instant: record index
	- dteday : date
	- season : season (1:springer, 2:summer, 3:fall, 4:winter)
	- yr : year (0: 2011, 1:2012)
	- mnth : month ( 1 to 12)
	- hr : hour (0 to 23)
	- holiday : weather 

## Schema and observation grain

In [3]:
day, hour = load_raw("day"), load_raw("hour")
for name, df in [("day.csv", day), ("hour.csv", hour)]:
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns | {df['dteday'].min().date()} to {df['dteday'].max().date()}")
display(day.head(3))
display(hour.head(3))

day.csv: 731 rows x 16 columns | 2011-01-01 to 2012-12-31
hour.csv: 17,379 rows x 17 columns | 2011-01-01 to 2012-12-31


,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,6,0,2,0.34,0.36,0.81,0.16,331,654,985
1,2,2011-01-02,1,0,1,0,0,0,2,0.36,0.35,0.70,0.25,131,670,801
2,3,2011-01-03,1,0,1,0,1,1,1,0.20,0.19,0.44,0.25,120,1229,1349


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.29,0.81,0.00,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.27,0.80,0.00,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.27,0.80,0.00,5,27,32


In [4]:
pd.DataFrame({"day.csv dtype": day.dtypes, "hour.csv dtype": hour.dtypes})

,day.csv dtype,hour.csv dtype
atemp,float64,float64
casual,int64,int64
cnt,int64,int64
dteday,datetime64[us],datetime64[us]
holiday,int64,int64
hr,NaN,int64
hum,float64,float64
instant,int64,int64
mnth,int64,int64
registered,int64,int64


**Grain:** one row of `day.csv` = one calendar day; one row of `hour.csv` = one clock hour of one day. Each has three count columns: `casual`, `registered` and `cnt` (total).

Grain rules followed throughout the project:
- **Intraday analysis uses `hour.csv` only. Daily, seasonal and trend analysis uses `day.csv` only.**
- The two tables are never appended or summed together, which would double-count rentals.
- `instant` is a row index, not a measure.
- Counts are **rental events**, not unique customers.

## Variable interpretation (from the Readme)
| Field | Meaning | Treatment |
|---|---|---|
| season | 1-4 season code | Categorical. The Readme labels must be checked against the dates (see notebook 02) |
| yr | 0 = 2011, 1 = 2012 | Categorical |
| weekday | day of week, **0 = Sunday** (verified in 02) | Categorical |
| holiday / workingday | flags | workingday = weekday and not holiday |
| weathersit | 1 clear ... 4 heavy rain/snow | **Ordinal category, not a continuous measure** |
| temp, atemp, hum, windspeed | normalized: divided by 41, 50, 100, 67 | Multiply back to physical units |

## Analytical questions and KPIs
| Area | Questions | KPIs (grain) |
|---|---|---|
| Demand & time | How are rentals distributed by hour, weekday, month, season and year? | Total rentals; mean/median per day; peak hour (per hour); YoY growth |
| Weather | How are temperature, humidity, wind and weather category associated with demand? | Mean per day by weather/temperature band; correlations |
| User segments | How do casual and registered users differ by time and weather? | Segment totals and shares; casual:registered ratio; hourly profiles |
| Operations | Which periods are consistently busy or quiet? | Demand tiers per (day type, hour) |

In [5]:
from src.analysis.kpis import headline_kpis, kpi_frame
from src.data.clean import load_processed
kpi_frame(headline_kpis(load_processed("day")))

,name,value,unit,grain,note
0,Total rentals,"3,292,679.00",rentals,period total,2011-01-01 to 2012-12-31
1,Registered rentals,"2,672,662.00",rentals,period total,
2,Casual rentals,"620,017.00",rentals,period total,
3,Registered share,0.81,share,ratio,
4,Casual share,0.19,share,ratio,
5,Casual-to-registered ratio,0.23,ratio,ratio,
6,Mean daily rentals,"4,504.35",rentals,per day,
7,Median daily rentals,"4,548.00",rentals,per day,
8,Max daily rentals,"8,714.00",rentals,per day,2012-09-15
9,Min daily rentals,22.00,rentals,per day,2012-10-29


## Conclusions & limitations
- Both files cover 731 days (2011-2012). `hour.csv` will be used for intraday patterns and `day.csv` for everything else.
- The dataset contains no station, inventory or trip-level information. The analysis **can describe rental demand but cannot measure availability, stockouts or unmet demand.**
- Next: notebook 02 assesses data quality and documents preparation decisions.